In [3]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [4]:
# Load Processed Data for Model Evaluation
BASE_PATH = "/content/drive/MyDrive/yellow_tripdata_2019"

from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.ml.evaluation import RegressionEvaluator
from pyspark.ml.tuning import CrossValidator, ParamGridBuilder
from pyspark.ml.regression import GBTRegressionModel
from pyspark.sql.types import *
import numpy as np

if 'spark' in locals() and spark is not None:
    try:
        spark.stop()
    except Exception as e:
        print(f"Warning: Could not stop existing SparkSession: {e}")

spark = SparkSession.builder \
    .appName("NYC Taxi Prediction") \
    .config("spark.driver.memory", "8g") \
    .config("spark.executor.memory", "8g") \
    .getOrCreate()


train_df = spark.read.parquet(f"{BASE_PATH}/data/final/train_2019")
val_df = spark.read.parquet(f"{BASE_PATH}/data/final/val_2019")
test_df = spark.read.parquet(f"{BASE_PATH}/data/final/test_2019")

print(f"Train (Jan-Oct): {train_df.count():,} rows ({train_df.count()/(train_df.count()+val_df.count()+test_df.count())*100:.1f}%")
print(f"Val (Nov):       {val_df.count():,} rows")
print(f"Test (Dec):      {test_df.count():,} rows")

Train (Jan-Oct): 68,360,825 rows (83.8%
Val (Nov):       6,603,163 rows
Test (Dec):      6,617,319 rows


In [5]:
# Train Initial Random Forest Model
from pyspark.ml.regression import RandomForestRegressor

rf = RandomForestRegressor(
    featuresCol="features",
    labelCol="fare_amount",
    numTrees=3,
    maxDepth=3,
    subsamplingRate=0.7
)

best_model = rf.fit(train_df)

In [8]:
# Evaluate Initial Model Performance
from pyspark.ml.evaluation import RegressionEvaluator
from pyspark.sql import Row
from pyspark.sql.functions import col

print("DISTRIBUTED REGRESSION METRICS\n")


test_preds = best_model.transform(test_df).cache()


evaluators = {
    "RMSE": RegressionEvaluator(
        labelCol="fare_amount",
        predictionCol="prediction",
        metricName="rmse"
    ),
    "MSE": RegressionEvaluator(
        labelCol="fare_amount",
        predictionCol="prediction",
        metricName="mse"
    ),
    "MAE": RegressionEvaluator(
        labelCol="fare_amount",
        predictionCol="prediction",
        metricName="mae"
    ),
    "R2": RegressionEvaluator(
        labelCol="fare_amount",
        predictionCol="prediction",
        metricName="r2"
    )
}


metrics = {}

for name, evaluator in evaluators.items():
    score = evaluator.evaluate(test_preds)
    rounded_score = __builtins__.round(score, 4)
    metrics[name] = rounded_score
    print(f"{name:<6}: {rounded_score}")


print("\nSUMMARY TABLE:")
summary_df = spark.createDataFrame([Row(**metrics)])
summary_df.show(truncate=False)


test_preds.unpersist()

DISTRIBUTED REGRESSION METRICS

RMSE  : 5.2451
MSE   : 27.5106
MAE   : 2.7246
R2    : 0.798

SUMMARY TABLE:
+------+-------+------+-----+
|RMSE  |MSE    |MAE   |R2   |
+------+-------+------+-----+
|5.2451|27.5106|2.7246|0.798|
+------+-------+------+-----+



DataFrame[features: vector, fare_amount: double, prediction: double]

In [9]:
# Analyze Temporal Performance Split
from pyspark.sql.functions import col, avg, lit

print("TEMPORAL SPLIT ANALYSIS:")
print("Temporal ordering confirmed:")
print("  Train: Jan-Oct 2019")
print("  Val:   Nov 2019")
print("  Test:  Dec 2019")


monthly_perf = (
    test_preds
        .withColumn("pickup_month", lit("2019-12"))
        .groupBy("pickup_month")
        .agg(
            avg(col("prediction") - col("fare_amount")).alias("mean_error"),
            avg(col("fare_amount")).alias("avg_fare")
        )
)


monthly_perf.show(truncate=False)

TEMPORAL SPLIT ANALYSIS:
Temporal ordering confirmed:
  Train: Jan-Oct 2019
  Val:   Nov 2019
  Test:  Dec 2019
+------------+--------------------+------------------+
|pickup_month|mean_error          |avg_fare          |
+------------+--------------------+------------------+
|2019-12     |-0.18536403825183898|13.293614698037121|
+------------+--------------------+------------------+



In [10]:
# Perform 5-Fold Stratified Cross-Validation
from pyspark.sql.functions import col, when
from pyspark.ml.evaluation import RegressionEvaluator
import numpy as np

print("5-FOLD STRATIFIED CROSS-VALIDATION")


fare_buckets = test_preds.withColumn(
    "fare_bucket",
    when(col("fare_amount") < 10, "low")
    .when(col("fare_amount") < 25, "medium")
    .otherwise("high")
)


print("Fare distribution:")
fare_buckets.groupBy("fare_bucket") \
    .count() \
    .orderBy("fare_bucket") \
    .show()


cv_evaluator = RegressionEvaluator(
    metricName="rmse",
    labelCol="fare_amount",
    predictionCol="prediction"
)

cv_scores = []


for i in range(5):

    fold = (
        fare_buckets
            .sample(withReplacement=False, fraction=0.2, seed=i)
            .select("features", "fare_amount")
            .cache()
    )

    fold_pred = best_model.transform(fold)

    rmse = cv_evaluator.evaluate(fold_pred)
    cv_scores.append(rmse)

    fold.unpersist()


print(f"Mean RMSE: ${np.mean(cv_scores):.2f} \u00b1 {np.std(cv_scores):.2f}")
print("CV Scores:", [f"{x:.2f}" for x in cv_scores])

5-FOLD STRATIFIED CROSS-VALIDATION
Fare distribution:
+-----------+-------+
|fare_bucket|  count|
+-----------+-------+
|       high| 690670|
|        low|3378344|
|     medium|2548305|
+-----------+-------+

Mean RMSE: $5.25 ± 0.03
CV Scores: ['5.29', '5.19', '5.24', '5.27', '5.26']


In [11]:
# Calculate Bootstrap Confidence Intervals for RMSE
from pyspark.ml.evaluation import RegressionEvaluator
import numpy as np

print("BOOTSTRAP CONFIDENCE INTERVALS")


test_preds_cached = (
    best_model
        .transform(test_df.coalesce(10))
        .select("prediction", "fare_amount")
        .cache()
)

test_preds_cached.count()


evaluator = RegressionEvaluator(
    labelCol="fare_amount",
    predictionCol="prediction",
    metricName="rmse"
)

def bootstrap_rmse_ci(n_bootstrap=50):

    rmse_scores = []

    for i in range(n_bootstrap):


        boot_sample = (
            test_preds_cached
                .sample(withReplacement=True, fraction=0.8, seed=i)
        )

        rmse = evaluator.evaluate(boot_sample)
        rmse_scores.append(rmse)


    rmse_scores = sorted(rmse_scores)

    lower_idx = int(0.025 * len(rmse_scores))
    upper_idx = int(0.975 * len(rmse_scores))

    ci_lower = rmse_scores[lower_idx]
    ci_upper = rmse_scores[upper_idx]

    return np.mean(rmse_scores), ci_lower, ci_upper



rmse_mean, rmse_ci_low, rmse_ci_high = bootstrap_rmse_ci(50)

print("RMSE Bootstrap Results:")
print(f"  Mean RMSE: ${rmse_mean:.2f}")
print(f"  95% CI:    ${rmse_ci_low:.2f} - ${rmse_ci_high:.2f}")
print(f"  CI Width:  ${rmse_ci_high - rmse_ci_low:.2f} (narrow = stable model)")


test_preds_cached.unpersist()

BOOTSTRAP CONFIDENCE INTERVALS
RMSE Bootstrap Results:
  Mean RMSE: $5.25
  95% CI:    $5.21 - $5.30
  CI Width:  $0.09 (narrow = stable model)


DataFrame[prediction: double, fare_amount: double]

In [12]:
# Align Model Performance with Business Metrics
from pyspark.sql.functions import col, avg, abs, count, when

print("BUSINESS METRICS ALIGNMENT")


test_with_profit = (
    test_preds
        .select("fare_amount", "prediction")
        .withColumn("driver_profit", col("fare_amount"))
        .withColumn("predicted_profit", col("prediction"))
)


business_metrics = (
    test_with_profit
        .agg(
            avg(col("driver_profit")).alias("avg_driver_profit"),
            avg(col("predicted_profit")).alias("avg_pred_profit"),
            avg(abs(col("driver_profit") - col("predicted_profit")))
                .alias("profit_forecast_error"),
            count(when(col("predicted_profit") > 0, True))
                .alias("profitable_trips_pred"),
            count(when(col("driver_profit") > 0, True))
                .alias("profitable_trips_actual")
        )
        .collect()[0]
)

print("NYC Taxi Driver Economics:")
print(f"Actual avg profit/trip:     ${business_metrics['avg_driver_profit']:.2f}")
print(f"Predicted avg profit/trip:  ${business_metrics['avg_pred_profit']:.2f}")
print(f"Profit forecast error:      ${business_metrics['profit_forecast_error']:.2f}")
print(f"Profitable trips actual:    {business_metrics['profitable_trips_actual']:,}")
print(f"Profitable trips predicted: {business_metrics['profitable_trips_pred']:,}")

BUSINESS METRICS ALIGNMENT
NYC Taxi Driver Economics:
Actual avg profit/trip:     $13.29
Predicted avg profit/trip:  $13.11
Profit forecast error:      $2.72
Profitable trips actual:    6,617,319
Profitable trips predicted: 6,617,319
